# Brand split

Splits `data/amazon_shoes_8_combined.parquet` into 6 buckets:

| gender | tier  | content |
|--------|-------|---------|
| Women  | top1  | #1 brand by ASIN count |
| Women  | top2  | #2 brand by ASIN count |
| Women  | other | long-tail brands (excluding well-known) |
| Men    | top1  | #1 brand by ASIN count |
| Men    | top2  | #2 brand by ASIN count |
| Men    | other | long-tail brands (excluding well-known) |

Each bucket is saved as `data/{gender}_{tier}_{brand_slug}/amazon_shoes_8_combined.parquet` and
is the input for a brand-specific run of the existing `data_preparation/` pipeline.

Well-known list is curated **per gender** so you can adjust without touching the other gender.

In [1]:
import pandas as pd
import re
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data'
INPUT_FILE   = DATA_ROOT / 'amazon_shoes_8_combined.parquet'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'INPUT_FILE   : {INPUT_FILE}')
assert INPUT_FILE.exists(), f'Missing input file: {INPUT_FILE}'

PROJECT_ROOT : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate
INPUT_FILE   : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/amazon_shoes_8_combined.parquet


In [2]:
df = pd.read_parquet(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

asin_df = df.drop_duplicates('ASIN')[['ASIN','brand','gender']]

print(f'Total rows         : {len(df):,}')
print(f'Unique ASINs       : {df["ASIN"].nunique():,}')
print(f'Unique brands      : {df["brand"].nunique():,}')
print()
print('Gender breakdown (ASIN-level):')
for g, n in asin_df['gender'].value_counts().items():
    print(f'  {g:<40} {n:>6}')

Total rows         : 1,008,930
Unique ASINs       : 7,761
Unique brands      : 664

Gender breakdown (ASIN-level):
  Women                                      5472
  Men                                        2279
  Costumes & Accessories                        9
  Shoe, Jewelry & Watch Accessories             1


## Well-known brand lists (per gender)

These lists define which brands get **excluded from the `other` bucket** so that `other` is purely long-tail.
Edit per gender — both lists seeded with the same curated default.

The audit cell below shows which brands actually appear in each gender's data so you can prune/add.

In [3]:
_DEFAULT_WELL_KNOWN = [
    'Nike',
    'adidas',
    'ASICS',
    'Reebok',
    'Skechers',
    'PUMA',
    'New Balance',
    'Converse',
    'Vans',
    'HEYDUDE',
    'OLUKAI',
    "Dr.Scholl's SHOES",
    'Crocs',
    'UGG',
    'Timberland',
    'Clarks',
    'Under Armour',
    'Amazon Essentials',
]

WOMEN_WELL_KNOWN = list(_DEFAULT_WELL_KNOWN)
MEN_WELL_KNOWN   = list(_DEFAULT_WELL_KNOWN)

WELL_KNOWN_BY_GENDER = {
    'Women': WOMEN_WELL_KNOWN,
    'Men':   MEN_WELL_KNOWN,
}

GENDERS_TO_PROCESS = ['Women', 'Men']

for g, lst in WELL_KNOWN_BY_GENDER.items():
    print(f'{g:<6}: {len(lst)} well-known brands')

Women : 18 well-known brands
Men   : 18 well-known brands


In [4]:
print('=' * 78)
print('WELL-KNOWN AUDIT')
print('=' * 78)
print('For each gender, list every name in that gender\'s well-known list and its')
print('ASIN count in the actual data. Names with 0 ASINs are not in this dataset.')
print()

for g in GENDERS_TO_PROCESS:
    g_brands = asin_df[asin_df['gender'] == g]['brand'].value_counts()
    wk_list  = WELL_KNOWN_BY_GENDER[g]

    present  = [b for b in wk_list if b in g_brands.index]
    missing  = [b for b in wk_list if b not in g_brands.index]
    extras   = sum(g_brands.get(b, 0) for b in present)

    print(f'--- {g} ---')
    print(f'  Well-known list size       : {len(wk_list)}')
    print(f'  Present in this gender     : {len(present)}  ({extras:,} ASINs total)')
    print(f'  Not in dataset (no-op)     : {len(missing)}')
    print()
    print(f'  {"brand":<25} {"ASINs in " + g:>15}')
    for b in wk_list:
        n = int(g_brands.get(b, 0))
        flag = '  (not in data)' if n == 0 else ''
        print(f'  {b:<25} {n:>15}{flag}')
    print()

WELL-KNOWN AUDIT
For each gender, list every name in that gender's well-known list and its
ASIN count in the actual data. Names with 0 ASINs are not in this dataset.

--- Women ---
  Well-known list size       : 18
  Present in this gender     : 11  (394 ASINs total)
  Not in dataset (no-op)     : 7

  brand                      ASINs in Women
  Nike                                   83
  adidas                                 37
  ASICS                                  47
  Reebok                                 40
  Skechers                               23
  PUMA                                   15
  New Balance                             0  (not in data)
  Converse                                0  (not in data)
  Vans                                    0  (not in data)
  HEYDUDE                                44
  OLUKAI                                 21
  Dr.Scholl's SHOES                      65
  Crocs                                   0  (not in data)
  UGG                 

## Compute top1 / top2 / other per gender

**Ranking rule:** top1 = brand with most ASINs in that gender, top2 = 2nd most.

**Other:** all remaining brands in that gender, **minus** anything in that gender's well-known list.
(If top1 or top2 happens to also be in the well-known list, it stays in its top tier — the well-known exclusion only applies to the `other` bucket.)

In [5]:
def slugify(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')

splits = {}

for g in GENDERS_TO_PROCESS:
    g_asin   = asin_df[asin_df['gender'] == g]
    counts   = g_asin['brand'].value_counts()

    top1_brand = counts.index[0]
    top2_brand = counts.index[1]

    wk_in_g = set(b for b in WELL_KNOWN_BY_GENDER[g] if b in counts.index)
    exclude_from_other = {top1_brand, top2_brand} | wk_in_g

    other_brands = [b for b in counts.index if b not in exclude_from_other]
    other_n_asin = int(g_asin[g_asin['brand'].isin(other_brands)]['ASIN'].nunique())

    splits[g] = {
        'top1_brand'  : top1_brand,
        'top1_n_asin' : int(counts.iloc[0]),
        'top2_brand'  : top2_brand,
        'top2_n_asin' : int(counts.iloc[1]),
        'well_known'  : sorted(wk_in_g),
        'wk_n_asin'   : int(g_asin[g_asin['brand'].isin(wk_in_g)]['ASIN'].nunique()),
        'other_brands': other_brands,
        'other_n_asin': other_n_asin,
    }

for g, s in splits.items():
    print('=' * 78)
    print(f'{g.upper()}')
    print('=' * 78)
    print(f'  top1   : {s["top1_brand"]:<25} {s["top1_n_asin"]:>5} ASINs')
    print(f'  top2   : {s["top2_brand"]:<25} {s["top2_n_asin"]:>5} ASINs')
    print(f'  well-known excluded from other : {len(s["well_known"])} brands ({s["wk_n_asin"]:,} ASINs)')
    for b in s['well_known']:
        flag = '  ← also top1' if b == s['top1_brand'] else ('  ← also top2' if b == s['top2_brand'] else '')
        n_b = int(asin_df[(asin_df['gender'] == g) & (asin_df['brand'] == b)].shape[0])
        print(f'    - {b:<25} {n_b:>5} ASINs{flag}')
    print(f'  other  : {len(s["other_brands"])} long-tail brands, {s["other_n_asin"]:,} ASINs')
    print()

WOMEN
  top1   : DREAM PAIRS                 447 ASINs
  top2   : IDIFU                       180 ASINs
  well-known excluded from other : 11 brands (394 ASINs)
    - ASICS                        47 ASINs
    - Amazon Essentials            16 ASINs
    - Dr.Scholl's SHOES            65 ASINs
    - HEYDUDE                      44 ASINs
    - Nike                         83 ASINs
    - OLUKAI                       21 ASINs
    - PUMA                         15 ASINs
    - Reebok                       40 ASINs
    - Skechers                     23 ASINs
    - Under Armour                  3 ASINs
    - adidas                       37 ASINs
  other  : 498 long-tail brands, 4,451 ASINs

MEN
  top1   : Bruno Marc                  165 ASINs
  top2   : Nike                        158 ASINs
  well-known excluded from other : 12 brands (542 ASINs)
    - ASICS                        61 ASINs
    - Amazon Essentials             4 ASINs
    - Dr.Scholl's SHOES             7 ASINs
    - HEYDUDE     

## Save the 6 buckets

In [6]:
summary_rows = []

for g, s in splits.items():
    g_df = df[df['gender'] == g]

    buckets = [
        ('top1',  s['top1_brand'],  g_df[g_df['brand'] == s['top1_brand']]),
        ('top2',  s['top2_brand'],  g_df[g_df['brand'] == s['top2_brand']]),
        ('other', None,             g_df[g_df['brand'].isin(s['other_brands'])]),
    ]

    for tier, brand_name, bucket_df in buckets:
        if brand_name is not None:
            folder_name = f'{g.lower()}_{tier}_{slugify(brand_name)}'
            label       = brand_name
        else:
            folder_name = f'{g.lower()}_{tier}'
            label       = f'{len(s["other_brands"])} long-tail brands'

        out_dir  = DATA_ROOT / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / 'amazon_shoes_8_combined.parquet'
        bucket_df.to_parquet(out_path, index=False)

        size_mb = out_path.stat().st_size / 1e6
        summary_rows.append({
            'folder'   : out_path.parent.relative_to(PROJECT_ROOT).as_posix(),
            'gender'   : g,
            'tier'     : tier,
            'brand'    : label,
            'rows'     : len(bucket_df),
            'asins'    : bucket_df['ASIN'].nunique(),
            'size_mb'  : round(size_mb, 1),
        })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
print()
print(f'Total buckets written: {len(summary)}')
print(f'Total rows written   : {summary["rows"].sum():,}')
print(f'Total ASINs written  : {summary["asins"].sum():,}  (vs source: {df["ASIN"].nunique():,})')
print(f'Difference           : {df["ASIN"].nunique() - summary["asins"].sum():,}  (= unused gender rows + well-known ASINs)')

                     folder gender  tier                brand   rows  asins  size_mb
data/women_top1_dream_pairs  Women  top1          DREAM PAIRS  58110    447      1.2
      data/women_top2_idifu  Women  top2                IDIFU  23400    180      0.4
           data/women_other  Women other 498 long-tail brands 578630   4451     10.5
   data/men_top1_bruno_marc    Men  top1           Bruno Marc  21450    165      0.5
         data/men_top2_nike    Men  top2                 Nike  20540    158      0.6
             data/men_other    Men other 234 long-tail brands 204360   1572      4.0

Total buckets written: 6
Total rows written   : 906,490
Total ASINs written  : 6,973  (vs source: 7,761)
Difference           : 788  (= unused gender rows + well-known ASINs)


## Next step

For each of the 6 output folders, run the existing `data_preparation/` pipeline pointing `INPUT_FILE` at that folder's parquet.

Suggested pattern in `data_preparation/00_part1_data_preparation.ipynb`:

```python
BRAND_BUCKET = 'women_top1_dream_pairs'   # change this for each run
INPUT_FILE   = DATA_ROOT / BRAND_BUCKET / 'amazon_shoes_8_combined.parquet'
DATA_DIR     = DATA_ROOT / BRAND_BUCKET / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
PRED_DIR     = DATA_ROOT / BRAND_BUCKET / 'predictions'
KEY_DIR      = DATA_ROOT / 'keys'
```